# Original Milan/Coppola model — independent Brunelli benchmark

This notebook simulates **only the reconstructed original Milan/Coppola model**.

It does not contain the final composed Brunelli model, TOMGRO/PyCrop state equations,
Brunelli-refined pest parameters, or Brunelli-fitted pest initial states.

The Brunelli dataset is used only for hourly temperature forcing and the common monitoring window.


In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
DATASET = HERE / "brunelli_final_dataset.csv"
OUT = HERE / "milan_outputs"
OUT.mkdir(exist_ok=True)
assert DATASET.exists()

raw = pd.read_csv(DATASET, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)
start = raw["datetime"].min()
raw["Day_float"] = (raw["datetime"] - start).dt.total_seconds()/86400.0
raw["Day"] = np.floor(raw["Day_float"]).astype(int)
N_HOURS = 70*24
raw = raw.iloc[:N_HOURS].copy()
hourly_temperature = raw["Temperature"].to_numpy(float)

EGG_LARVA_DAYS = np.array([0,6,13,20,27,34,41,48,55,62,69], dtype=int)
ADULT_DAYS = np.array([0,7,14,21,28,36,42,48,55,62,69], dtype=int)

NORMALIZATION=False
AMPLITUDE_SCALING=False
SMOOTHING=False
POSTHOC_SHIFT=False
TRAP_Q=False


## Source-only Milan parameterisation

The main benchmark uses the source Milan biological parameterisation and source-defined adult-only start.
The audited reconstruction uses `beta=0.0012`, `delta=35`, and crop cardinal temperatures `(5,22,32) °C`
because these choices reproduce the crop scale reported in Coppola's Chapter 4 while avoiding transfer
of any Brunelli-refined parameter.


In [2]:
MILAN = {
    "mu_E":0.24, "mu_L":0.22, "mu_P":0.04, "mu_A":0.27,
    "zeta_A":10.28,
    "a2":57.72, "b2":0.10,
    "a3":73.78, "b3":0.09,
    "fertility_a":290.58, "fertility_b":1.57, "fertility_c":20.64,
    "T_pest_min":4.90, "T_pest_max":32.7,
    "alpha":6.5e-4, "delta":35.0,
    "epsilon_max":0.55, "chi":4e-3,
    "Omega":1.0/35.0, "nu":5e-4,
    "sigma":0.1, "rho_max":0.144, "beta":0.0012, "iota":3000.0,
    "T_crop_min":5.0, "T_crop_opt":22.0, "T_crop_max":32.0,
}
MILAN_IC = {"F0":18.0, "E0":0.0, "L0":0.0, "P0":0.0, "A0":2.0}
W=1.0
Z=0.0

pd.DataFrame(
    [{"Parameter":k,"Value":v} for k,v in MILAN.items()]
    + [{"Parameter":k,"Value":v} for k,v in MILAN_IC.items()]
    + [{"Parameter":"w","Value":W},{"Parameter":"z","Value":Z}]
).to_csv(OUT/"milan_run_configuration.csv", index=False)


In [3]:
def total_fertility(T,p=MILAN):
    T=float(T)
    if not (p["T_pest_min"] < T < p["T_pest_max"]):
        return 0.0
    return (p["fertility_a"]*(T-p["T_pest_min"])**p["fertility_b"]*(p["T_pest_max"]-T)
            /((p["fertility_c"]-p["T_pest_min"])**p["fertility_b"]
              *(p["T_pest_max"]-p["T_pest_min"])))

def phi(T,p=MILAN):
    return total_fertility(T,p)/(2.0*p["zeta_A"])

def crop_rho_temperature(T,p=MILAN):
    T=float(T)
    Tmin,Topt,Tmax=p["T_crop_min"],p["T_crop_opt"],p["T_crop_max"]
    if not (Tmin < T < Tmax):
        return 0.0
    left=(T-Tmin)/(Topt-Tmin)
    right=(Tmax-T)/(Tmax-Topt)
    exponent=(Tmax-Topt)/(Topt-Tmin)
    return p["rho_max"]*left*right**exponent

def rhs(state,T,p=MILAN):
    F,E,L,P,A=np.maximum(np.asarray(state,dtype=float),0.0)
    feeding=p["alpha"]*F/(F+p["delta"]) if F>0 else 0.0
    zL=p["chi"]/(p["epsilon_max"]*W*feeding) if feeding>0 else 1e12
    zE=p["a2"]*math.exp(-p["b2"]*T)
    zP=p["a3"]*math.exp(-p["b3"]*T)
    hatch=E/zE
    free_spots=max(0.0,F*p["Omega"]/p["nu"]-L)
    larval_inflow=min(hatch,free_spots)
    predation=Z*E/(p["iota"]+E) if E>0 else 0.0
    rho=crop_rho_temperature(T,p)*W*math.exp(-p["sigma"]*L/F) if F>0 else 0.0
    return np.array([
        rho*F-p["beta"]*F*F-feeding*L,
        phi(T,p)*A-hatch-p["mu_E"]*E-predation,
        larval_inflow-L/zL-p["mu_L"]*L,
        L/zL-P/zP-p["mu_P"]*P,
        P/zP-p["mu_A"]*A,
    ],dtype=float)


In [4]:
def simulate():
    dt=1.0/24.0
    state=np.array([MILAN_IC[k] for k in ["F0","E0","L0","P0","A0"]],dtype=float)
    states=[]
    cumulative=[]
    cumA=0.0
    for T in hourly_temperature:
        states.append(state.copy())
        cumulative.append(cumA)
        k1=rhs(state,T)
        k2=rhs(np.maximum(state+0.5*dt*k1,0.0),T)
        k3=rhs(np.maximum(state+0.5*dt*k2,0.0),T)
        k4=rhs(np.maximum(state+dt*k3,0.0),T)
        state=np.maximum(state+dt*(k1+2*k2+2*k3+k4)/6.0,0.0)
        cumA += state[4]*dt
    return np.asarray(states),np.asarray(cumulative)

hourly_states,cumulative=simulate()

hourly_df=pd.DataFrame(hourly_states,columns=["F","Egg","Larva","Pupa","Adult"])
hourly_df.insert(0,"Hour",np.arange(len(hourly_df)))
hourly_df["Day_float"]=hourly_df["Hour"]/24.0
hourly_df["Cumulative_Adult_Exposure_at_hour_start"]=cumulative
hourly_df.to_csv(OUT/"milan_hourly_states.csv",index=False)

daily=hourly_states[np.arange(0,N_HOURS,24)]
daily_df=pd.DataFrame(daily,columns=["F","Egg","Larva","Pupa","Adult"])
daily_df.insert(0,"Day",np.arange(len(daily_df)))
daily_df.to_csv(OUT/"milan_daily_states.csv",index=False)

survey=pd.DataFrame({
    "Day":EGG_LARVA_DAYS,
    "Milan_Egg":daily[EGG_LARVA_DAYS,1],
    "Milan_Larva":daily[EGG_LARVA_DAYS,2],
})
survey.to_csv(OUT/"milan_egg_larva_survey_predictions.csv",index=False)

adult_interval=np.diff(cumulative[ADULT_DAYS*24])
adult=pd.DataFrame({
    "Interval_start_day":ADULT_DAYS[:-1],
    "Interval_end_day":ADULT_DAYS[1:],
    "Milan_Adult_interval_exposure":adult_interval,
})
adult.to_csv(OUT/"milan_adult_interval_predictions.csv",index=False)

display(survey)
display(adult)


,Day,Milan_Egg,Milan_Larva
0,0,0.000000,0.000000
1,6,5.266899,5.837333
2,13,5.331337,3.948262
3,20,5.825035,3.687911
4,27,4.850419,4.910461
5,34,7.033154,5.766269
6,41,10.219235,5.960486
7,48,11.182570,6.738075
8,55,12.285260,6.848744
9,62,14.295931,8.160020


,Interval_start_day,Interval_end_day,Milan_Adult_interval_exposure
0,0,7,6.619473
1,7,14,3.078947
2,14,21,3.226818
3,21,28,4.460047
4,28,36,6.628075
5,36,42,4.755579
6,42,48,5.353465
7,48,55,7.084272
8,55,62,7.600004
9,62,69,8.921088


In [5]:
assert MILAN["a2"]==57.72
assert MILAN["b2"]==0.10
assert MILAN["mu_E"]==0.24
assert MILAN["nu"]==5e-4
assert MILAN_IC=={"F0":18.0,"E0":0.0,"L0":0.0,"P0":0.0,"A0":2.0}
assert not any([NORMALIZATION,AMPLITUDE_SCALING,SMOOTHING,POSTHOC_SHIFT,TRAP_Q])
print("MILAN NOTEBOOK AUDIT PASSED")


MILAN NOTEBOOK AUDIT PASSED
